# Graph RAG ハンズオン

## 1. 環境変数の読み込み

In [1]:
from dotenv import load_dotenv

# .envファイルの環境変数を読み込む
load_dotenv()

True

## 2. Graph RAG (ナレッジグラフ手動作成)

### ナレッジグラフ作成

In [2]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph()

# グラフデータベースの既存の知識グラフを削除 (※消したくない場合はコメントアウト)
graph.query("""MATCH (n)
DETACH DELETE (n)""")

sar_graph_query = """CREATE (otani:Person {name: '大谷'})
CREATE (hirayama:Person {name: '平山'})
CREATE (rag:Tech {name: 'RAG'})
CREATE (gcp:Tech {name: 'GCP'})
CREATE (tableau:Tech {name: 'Tableau'})

CREATE (otani)-[:LIKE]->(hirayama)
CREATE (otani)-[:HAS_SKILL]->(gcp)
CREATE (otani)-[:INTERESTED_IN]->(rag)
CREATE (hirayama)-[:LIKE]->(otani)
CREATE (hirayama)-[:HAS_SKILL]->(tableau)
"""

In [3]:
graph.query(sar_graph_query)

[]

![SARナレッジグラフ](../../img/sar_graph.png)

### グラフデータベース スキーマ確認 + クエリ実行

In [4]:
graph = Neo4jGraph()
print(graph.get_schema)

Node properties:
Person {name: STRING}
Tech {name: STRING}
Relationship properties:

The relationships:
(:Person)-[:LIKE]->(:Person)
(:Person)-[:HAS_SKILL]->(:Tech)
(:Person)-[:INTERESTED_IN]->(:Tech)


In [5]:
graph.query("""MATCH (n) -[r]-> (m)
RETURN n, r, m""")

[{'n': {'name': '大谷'},
  'r': ({'name': '大谷'}, 'LIKE', {'name': '平山'}),
  'm': {'name': '平山'}},
 {'n': {'name': '大谷'},
  'r': ({'name': '大谷'}, 'HAS_SKILL', {'name': 'GCP'}),
  'm': {'name': 'GCP'}},
 {'n': {'name': '大谷'},
  'r': ({'name': '大谷'}, 'INTERESTED_IN', {'name': 'RAG'}),
  'm': {'name': 'RAG'}},
 {'n': {'name': '平山'},
  'r': ({'name': '平山'}, 'LIKE', {'name': '大谷'}),
  'm': {'name': '大谷'}},
 {'n': {'name': '平山'},
  'r': ({'name': '平山'}, 'HAS_SKILL', {'name': 'Tableau'}),
  'm': {'name': 'Tableau'}}]

### LangChain × Graph RAG

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [7]:
# Geminiモデルの選択
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
# Cypher言語をLLMに出力させるためのプロンプトテンプレート
cypher_template = """Neo4jの以下のグラフスキーマに基づいて、ユーザの質問に答えるCypherクエリを書いてください。:
{schema}
質問: {question}
Cypherクエリ:"""

In [ ]:
# Cypher言語をLLMに出力させるためのプロンプト本体
cypher_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "入力された質問をCypherクエリに変換してください。クエリ以外は生成しないでください。",
        ),
        ("human", cypher_template),
    ]
)

In [ ]:
# プロンプトの動作確認
result = cypher_prompt.invoke({
    "schema": "test_schema_contents",
    "question": "test_question"
})

print(result.to_string())

System: 入力された質問をCypherクエリに変換してください。クエリ以外は生成しないでください。
Human: Neo4jの以下のグラフスキーマに基づいて、ユーザの質問に答えるCypherクエリを書いてください。:
test_schema_contents
質問: test_question
Cypherクエリ:


In [34]:
# 質問文からCypher言語を出力するチェイン
chain_generate_query = (
    RunnablePassthrough.assign(schema=lambda _: graph.get_schema)
    | cypher_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 動作確認 (質問 → Cypher言語)

# question = input("question > ")
question = "大谷さんが好きな人は誰ですか？"

generated_query = chain_generate_query.invoke({"question": question})
print(generated_query)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


MATCH (p1:Person {name: '大谷'})-[:LIKE]->(p2:Person)
RETURN p2.name


In [ ]:
# 質問に対する回答をさせるためのプロンプトテンプレート
response_template = """質問、Cypherクエリ、およびクエリ実行結果に基づいて、自然言語で回答を書いてください。:
質問: {question}
Cypherクエリ: {query}
クエリ実行結果: {response}"""

In [ ]:
# 質問に対する回答をさせるためのプロンプト本体
response_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "入力された質問、クエリ、クエリ実行結果をもとに、自然言語の答えに変換してください。",
        ),
        ("human", response_template),
    ]
)

### チェイン中の入力値の推移

```
1. {"question": XXXXX}
↓
2. {"question": XXXXX, "query": chain_generate_query({"question": XXXXX})}
↓
chain_generate_query({"question": XXXXX}) = YYYYY (Cypherクエリ) とすると
↓
3. {"question": XXXXX, "query": YYYYY, "response", graph.query(YYYYY)}
↓
graph.query(YYYYY) = ZZZZZ (Cypherクエリの実行結果) とすると
↓
4. {"question": XXXXX, "query": YYYYY, "response", ZZZZZ}
```

In [ ]:
# 質問文から回答を出力するチェイン
chain = (
    RunnablePassthrough.assign(query=chain_generate_query)  # 
    | RunnablePassthrough.assign(
        response=lambda x: graph.query(x["query"]),
    )
    | response_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 動作確認 (質問 → 回答)

# question = input("question > ")
question = "大谷さんが好きな人は誰ですか？"

result = chain.invoke({"question": question})
print(result)

大谷さんが好きな人は平山さんです。


## 3. Graph RAG (ナレッジグラフ自動作成)

In [28]:
# グラフデータベースの既存の知識グラフを削除 (※消したくない場合はコメントアウト)
graph.query("""MATCH (n)
DETACH DELETE (n)""")

[]

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

# テキストをナレッジグラフ構造に自動で変換するツール
llm_transformaer = LLMGraphTransformer(llm=llm)

In [26]:
from langchain_core.documents import Document

text = """社員技術プロフィール
平山さん:
・実務ではTableauを主に使用している
・最近はWordPressを使用し、フロントエンドエンジニアとして活躍中
大谷さん
・実務では主にGCPを使用している
・Microsoft製品に強い恨みを持っているため、使用不可
・最近はRAGについて学習中
林さん
・社内の研修担当者
・最近は「データ基盤構築」「RAG」「dbt」の技術のキャッチアップでパンクしそう
"""

In [ ]:
# ノードとリレーションシップを自動作成
documents = [Document(page_content=text)]
graph_documents = llm_transformaer.convert_to_graph_documents(documents)

In [ ]:
# グラフデータベースにナレッジグラフを追加
graph.add_graph_documents(graph_documents)

In [31]:
graph = Neo4jGraph()
print(graph.get_schema)

Node properties:
Person {id: STRING}
Technology {id: STRING}
Organization {id: STRING}
Relationship properties:

The relationships:
(:Person)-[:USES]->(:Technology)
(:Person)-[:LEARNS]->(:Technology)
(:Person)-[:DISLIKES]->(:Organization)


![SARナレッジグラフ](../../img/sar_profile.png)

In [33]:
# question = input("question > ")
question = "大谷さんにMicrosoftの案件を任せても大丈夫か"

result = chain.invoke({"question": question})
print(result)

クエリ実行結果から、大谷さんはMicrosoftに対して否定的な感情（DISLIKES）を持っていることが確認されています。そのため、大谷さんにMicrosoftの案件を任せるのは避けた方が良いかもしれません。


In [ ]:
# VectorDBとは異なり、GraphDBではドキュメント全体に関する問い合わせが可能
# VectorDB → 質問との類似度での検索になるため、単純な質問文と回答パターンの意味の近さでしけ検索ができない
# GraphDB → 適切なCypherクエリさえ生成できれば実行可能

# question = input("question > ")
question = "このデータベースには平山さんに関する情報はありますか？"

result = chain.invoke({"question": question})
print(result)

はい、このデータベースには平山さんに関する情報が存在します。
